# Bayesian Model Comparison

## Overview

Bayesian model comparison evaluates how well models predict new data, properly accounting for model complexity. Unlike AIC/BIC (which are approximations), fully Bayesian methods compute the marginal likelihood or use leave-one-out cross-validation.

**Methods:**

| Method | Measures | Notes |
|---|---|---|
| WAIC | Widely Applicable IC | Asymptotically equivalent to LOO-CV |
| PSIS-LOO-CV | Leave-one-out CV | Preferred; Pareto-smoothed importance sampling |
| Bayes Factor | P(data \| M1) / P(data \| M2) | Prior-sensitive; hard to compute |
| Posterior predictive check | Does model generate plausible data? | Visual validation |

PSIS-LOO-CV (via ArviZ `az.loo()`) is the current recommended approach. It estimates out-of-sample predictive accuracy without refitting the model.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm

rng = np.random.default_rng(42)
n = 80
elevation = rng.uniform(50, 400, n)
nitrate   = rng.gamma(2, 2, n)
treatment = rng.choice([0,1], n)
richness  = 25 - 0.03*elevation - 0.7*nitrate + 2.5*treatment + rng.normal(0,3.5,n)
df = pd.DataFrame({"richness":richness,"elevation":elevation,
                   "nitrate":nitrate,"treatment":treatment})

---
## WAIC and LOO-CV with PyMC + ArviZ

In [ ]:
try:
    import pymc as pm
    import arviz as az
    models_traces = {}
    elev_s = (elevation-elevation.mean())/elevation.std()
    nitr_s = (nitrate-nitrate.mean())/nitrate.std()
    # Model 1: intercept only
    with pm.Model() as m0:
        intercept = pm.Normal("intercept", 20, 10)
        sigma = pm.HalfNormal("sigma", 8)
        pm.Normal("y", intercept, sigma, observed=richness)
        models_traces["M0_intercept"] = pm.sample(
            1000, tune=500, chains=2, random_seed=42, progressbar=False,
            idata_kwargs={"log_likelihood":True})
    # Model 2: full model
    with pm.Model() as m1:
        intercept = pm.Normal("intercept", 20, 10)
        b_e = pm.Normal("b_elev", 0, 5)
        b_n = pm.Normal("b_nitr", 0, 5)
        b_t = pm.Normal("b_treat", 0, 5)
        sigma = pm.HalfNormal("sigma", 8)
        mu = intercept + b_e*elev_s + b_n*nitr_s + b_t*treatment
        pm.Normal("y", mu, sigma, observed=richness)
        models_traces["M1_full"] = pm.sample(
            1000, tune=500, chains=2, random_seed=42, progressbar=False,
            idata_kwargs={"log_likelihood":True})
    # LOO comparison
    comp = az.compare({k: az.loo(v) for k,v in models_traces.items()},
                      method="stacking")
    print(comp)
    az.plot_compare(comp, insample_dev=False)
    plt.tight_layout(); plt.show()
except ImportError:
    print("PyMC/ArviZ not installed: pip install pymc arviz")
    print("LOO-CV comparison via ArviZ:")
    print("  az.compare() ranks models by expected log predictive density (ELPD)")
    print("  Higher ELPD = better out-of-sample prediction")
    print("  dELPD / SE ratio > 2 indicates meaningful difference")

---
## AIC and BIC as Frequentist Approximations

In [ ]:
models_ols = {
    "Intercept only": smf.ols("richness ~ 1", df).fit(),
    "Elevation":      smf.ols("richness ~ elevation", df).fit(),
    "Full model":     smf.ols("richness ~ elevation + nitrate + treatment", df).fit(),
    "Overfit (poly)": smf.ols("richness ~ elevation + I(elevation**2) + nitrate + I(nitrate**2) + treatment", df).fit(),
}
rows = []
for name, m in models_ols.items():
    rows.append({"Model":name,"AIC":m.aic,"BIC":m.bic,
                 "R2":m.rsquared,"R2_adj":m.rsquared_adj,"n_params":m.df_model+1})
result_df = pd.DataFrame(rows).set_index("Model")
print(result_df.round(2))
print("\nDelta AIC from best model:")
best_aic = result_df["AIC"].min()
print((result_df["AIC"] - best_aic).round(2))

---
## Posterior Predictive Checks

In [ ]:
# Simulate: draw coefficients from approximate posterior, generate replicated datasets
objective_model = smf.ols("richness ~ elevation + nitrate + treatment", df).fit()
coef_samples = rng.multivariate_normal(
    objective_model.params,
    objective_model.cov_params(),
    size=500
)
sigma_samples = np.sqrt(objective_model.mse_resid) * rng.chisquare(
    objective_model.df_resid, 500) / objective_model.df_resid
X_pp = sm.add_constant(df[["elevation","nitrate","treatment"]])
fig, axes = plt.subplots(1,2,figsize=(12,4))
# Histogram PPC
for coef, sig in zip(coef_samples[:100], sigma_samples[:100]):
    y_rep = rng.normal(X_pp.values @ coef, sig)
    axes[0].hist(y_rep, bins=25, density=True, alpha=0.05, color="steelblue")
axes[0].hist(richness, bins=25, density=True, alpha=0.8,
             color="#e74c3c", label="Observed", histtype="step", lw=2)
axes[0].set_title("Posterior Predictive Check: Replicated vs Observed")
axes[0].legend()
# Mean PPC
rep_means = [rng.normal(X_pp.values @ c, s).mean() for c,s in zip(coef_samples,sigma_samples)]
axes[1].hist(rep_means, bins=40, density=True, color="steelblue", alpha=0.7)
axes[1].axvline(richness.mean(), color="#e74c3c", lw=2, label=f"Observed mean={richness.mean():.2f}")
axes[1].set_title("PPC: Distribution of Replicated Means")
axes[1].legend(); plt.tight_layout(); plt.show()

---
## Bayes Factors (Conceptual)

In [ ]:
# Bayes Factor: ratio of marginal likelihoods -- hard to compute directly
# Demonstrate via Savage-Dickey density ratio for point null H0: beta_treatment = 0
# Prior: N(0, 5); approximate posterior from OLS
full_model = smf.ols("richness ~ elevation + nitrate + treatment", df).fit()
b_treat_post_mean = full_model.params["treatment"]
b_treat_post_se   = full_model.bse["treatment"]
prior_sd = 5.0
# Savage-Dickey: BF = p(theta=0 | M1) / p(theta=0 | prior)
prior_at_zero    = stats.norm.pdf(0, 0, prior_sd)
posterior_at_zero = stats.norm.pdf(0, b_treat_post_mean, b_treat_post_se)
BF10 = prior_at_zero / posterior_at_zero
print(f"Savage-Dickey BF10 (H1:treatment_effect vs H0:no_effect): {BF10:.2f}")
print(f"Interpretation: data are {BF10:.1f}x more likely under H1 than H0")
print("\nJeffreys scale:")
print("  BF < 3:   weak evidence")
print("  BF 3-10:  moderate evidence")
print("  BF 10-30: strong evidence")
print("  BF > 100: decisive evidence")

---

## Common Pitfalls

**1. Using AIC/BIC to compare models with different likelihoods or data subsets**  
AIC and BIC are only comparable across models fit to exactly the same data with the same likelihood function. Models fit by REML vs ML, or to different subsets, cannot be compared with AIC/BIC.

**2. Treating lower AIC as always better without checking absolute fit**  
AIC ranks models relative to each other but says nothing about absolute fit. A model with the lowest AIC may still generate predictions that look nothing like the data. Always follow model comparison with posterior predictive checks.

**3. Interpreting PSIS-LOO warnings about high Pareto k values**  
Pareto k > 0.7 for specific observations indicates those observations are highly influential — the LOO estimate for those points is unreliable. Investigate whether those observations are outliers or whether the model is misspecified.

**4. Comparing Bayes Factors across different prior specifications**  
Bayes Factors depend heavily on prior choice in a way that ELPD-based metrics do not. A Bayes Factor computed with a N(0,5) prior is not comparable to one computed with N(0,2). Always report the prior used when reporting Bayes Factors.

**5. Skipping posterior predictive checks in favour of model comparison metrics alone**  
A model may win a comparison tournament while still being badly misspecified. ELPD and BF measure relative performance; PPCs check whether the winning model generates plausible data. Both are necessary.
---
*python_methods_library - Samantha McGarrigle*